# 02 - Sampling Estratificado (`merged-after-rework`)

Ejecuta el muestreo real con `exploration/aidev/sampling/stratified_sampler.py` y genera los artifacts canónicos:
- `exploration/aidev/sampling/outputs/merged_after_rework_sample.csv`
- `exploration/aidev/sampling/outputs/merged_after_rework_sample_summary.json`

Sobrescribe outputs por defecto (reproducible por seed).


In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PY = PROJECT_ROOT / '.venv' / 'bin' / 'python'
SAMPLER = PROJECT_ROOT / 'exploration/aidev/sampling/stratified_sampler.py'

# Parámetros del estudio (editables)
SAMPLE_SIZE = 300
MIN_PER_STRATUM = 3
SEED = 20260510

OUT_CSV = PROJECT_ROOT / 'exploration/aidev/sampling/outputs/merged_after_rework_sample.csv'
OUT_SUMMARY = PROJECT_ROOT / 'exploration/aidev/sampling/outputs/merged_after_rework_sample_summary.json'

def run(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    print('[cmd]', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=True, text=True, capture_output=True)

print('[info] OUT_CSV=', OUT_CSV)
print('[info] OUT_SUMMARY=', OUT_SUMMARY)


In [ ]:
print('\n[step] Ejecutando muestreo estratificado (consultando Parquet remotos)...')
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_SUMMARY.parent.mkdir(parents=True, exist_ok=True)
cmd = [
    str(PY), str(SAMPLER),
    '--source', 'aidev',
    '--population-mode', 'merged-after-rework',
    '--sample-size', str(SAMPLE_SIZE),
    '--min-per-stratum', str(MIN_PER_STRATUM),
    '--seed', str(SEED),
    '--output-csv', str(OUT_CSV),
    '--summary-json', str(OUT_SUMMARY),
]
run(cmd)
print('[ok] sampling completado')


In [ ]:
print('\n[step] Validación rápida')
summary = json.loads(OUT_SUMMARY.read_text(encoding='utf-8'))
print('[info] population_size=', summary.get('population_size'))
print('[info] sample_size=', summary.get('sample_size'))
print('[info] quotas=', summary.get('quotas'))
row_count = sum(1 for _ in OUT_CSV.open('r', encoding='utf-8')) - 1
print('[info] csv_rows=', row_count)
if row_count != SAMPLE_SIZE:
    print('[warn] csv_rows != SAMPLE_SIZE (revisar min_per_stratum/strata)')
else:
    print('[ok] tamaño de muestra correcto')
